In [97]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np

In [ ]:
parent_folder_path = '/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/MODIS_L2'
os.chdir(parent_folder_path)

cloud_file = "MOD06_L2"
geo_file = "MOD03"

years = ["2020"]
days  = ["{:03d}".format(num) for num in range(1,2)] # keep day 366, ignore in nonleap years
hours = ["{:02d}".format(num) for num in range(0, 2)]
mins  = ["{:02d}".format(num) for num in range(0, 60, 5)]

km5vars = ["Cloud_Fraction", 
        "Cloud_Fraction_Nadir",         
        "Cloud_Fraction_Night",         
        "Cloud_Fraction_Nadir_Night",   
        "Cloud_Fraction_Day",           
        "Cloud_Fraction_Nadir_Day"]

km1vars = ["Cloud_Optical_Thickness",             
    "Cloud_Optical_Thickness_PCL",         
    "Cloud_Optical_Thickness_16",          
    "Cloud_Optical_Thickness_16_PCL",      
    "Cloud_Optical_Thickness_37",          
    "Cloud_Optical_Thickness_37_PCL",          
    "Cloud_Optical_Thickness_1621",        
    "Cloud_Optical_Thickness_1621_PCL",
    "Cloud_Optical_Thickness_Uncertainty",       
    "Cloud_Optical_Thickness_Uncertainty_16",    
    "Cloud_Optical_Thickness_Uncertainty_37",    
    "Cloud_Optical_Thickness_Uncertainty_1621"]

# containers for all data
df_list_1km = []
df_list_5km = []

# although we only download 20 days once a time,
# but we iterating over all the days to avoid manual operation errors
for year in years:
    for day in days:

        km1_daily = []
        km5_daily = []

        for hour in hours:
            for min in mins:
                cloud_path = glob.glob(cloud_file + "/" + year + "/" + day + "/" + cloud_file + ".A" + year + day + "." + hour + min + "*.hdf")
                geo_path = glob.glob(geo_file + "/" + year + "/" + day + "/" + geo_file + ".A" + year + day + "." + hour + min + "*.hdf")

                if len(cloud_path) == 0 or len(geo_path) == 0: #no files exist, go next
                    continue
                else:
                    # read data file
                    cloud = xr.open_dataset(cloud_path[0], engine="netcdf4", decode_times=False, mask_and_scale=True)

                    # read geolocation file
                    geo = xr.open_dataset(geo_path[0], engine="netcdf4", decode_times=False, mask_and_scale=True)
                    
                    ### ————1km————
                    # extract 1 km lat/lon
                    lat_1km = geo["Latitude"].data     # (2030,1354)
                    lon_1km = geo["Longitude"].data

                    # 1km data
                    one_km_dict = {
                    "lat": lat_1km.ravel(),
                    "lon": lon_1km.ravel(),
                    "year": year,
                    "day": day,
                    "hour": hour,
                    "minute": min
                    }

                    # load each 1 km variable (same shape as lat/lon)
                    for var in km1vars:
                        if cloud[var].shape == lat_1km.shape and cloud[var].shape == lon_1km.shape:
                            try:
                                one_km_dict[var] = cloud[var].data.ravel()
                            except Exception as inst:
                                print(year,day,hour,min,inst)
                        else:
                            print(year,day,hour,min,"1km shape unmatch")
                            print(cloud[var].shape)


                    df_1km_min = pd.DataFrame(one_km_dict)
                    km1_daily.append(df_1km_min)

                    ### ————5km————
                    # extract 5 km lat/lon
                    lat_5km = cloud["Latitude"].data     # (406,270)
                    lon_5km = cloud["Longitude"].data

                    # 5km data
                    five_km_dict = {
                    "lat": lat_5km.ravel(),
                    "lon": lon_5km.ravel(),
                    "year": year,
                    "day": day,
                    "hour": hour,
                    "minute": min
                    }

                    # load each 5 km variable (same shape as lat/lon)
                    for var in km5vars:
                        if cloud[var].shape == lat_5km.shape and cloud[var].shape == lon_5km.shape:
                            try:
                                five_km_dict[var] = cloud[var].data.ravel()
                            except Exception as inst:
                                print(year,day,hour,min,inst)
                        else:
                            print(year,day,hour,min,"5km shape unmatch")
                            print(cloud[var].shape)

                    df_5km_min = pd.DataFrame(five_km_dict)
                    km5_daily.append(df_5km_min)
        
        df_km5_daily  = pd.concat(km5_daily)
        df_km1_daily  = pd.concat(km1_daily)


                    
       

In [125]:
df_km5_daily

,lat,lon,year,day,hour,minute,Cloud_Fraction,Cloud_Fraction_Nadir,Cloud_Fraction_Night,Cloud_Fraction_Nadir_Night,Cloud_Fraction_Day,Cloud_Fraction_Nadir_Day
0,8.959794,-13.194514,2020,001,00,00,0.80,NaN,0.80,NaN,NaN,NaN
1,8.935727,-13.405244,2020,001,00,00,0.68,NaN,0.68,NaN,NaN,NaN
2,8.912421,-13.608292,2020,001,00,00,0.20,NaN,0.20,NaN,NaN,NaN
3,8.889836,-13.804123,2020,001,00,00,0.80,NaN,0.80,NaN,NaN,NaN
4,8.867930,-13.993201,2020,001,00,00,0.92,NaN,0.92,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
109615,71.544876,-125.596184,2020,001,01,55,1.00,NaN,1.00,NaN,NaN,NaN
109616,71.368904,-125.766380,2020,001,01,55,1.00,NaN,1.00,NaN,NaN,NaN
109617,71.186676,-125.939240,2020,001,01,55,1.00,NaN,1.00,NaN,NaN,NaN
109618,70.997765,-126.114914,2020,001,01,55,1.00,NaN,1.00,NaN,NaN,NaN


In [ ]:
parent_folder_path = '/Users/anora/Library/Cloudatatorage/Dropbox-TeamMG/Wanru Wu/Cloudataeeding_Anora/MODIS_L2'
os.chdir(parent_folder_path)

data_file = "MOD06_L2"
geo_file = "MOD03"

years = ["2020","2021","2022","2023","2024","2025"]
days  = ["{:03d}".format(num) for num in range(1,367)] # keep day 366, ignore in nonleap years
hours = ["{:02d}".format(num) for num in range(0, 24)]
mins  = ["{:02d}".format(num) for num in range(0, 60, 5)]



SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='? (3256079248.py, line 24)

In [111]:
import xarray as xr

path1 = "MOD06_L2/2020/001/MOD06_L2.A2020001.0000.061.2020002234145.hdf"
ds = xr.open_dataset(path1, engine="netcdf4",decode_times=False,mask_and_scale=True)

if not np.any(ds['Cloud_Optical_Thickness'].data): # This will be True for an empty array
    print("The NumPy array is empty.")
else:
    print("The NumPy array is not empty.")

The NumPy array is not empty.


In [61]:
import xarray as xr
import warnings

warnings.filterwarnings("ignore") 

path2 = "/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/MODIS_L2/MOD03/2020/001/MOD03.A2020001.0000.061.2020001070344.hdf"
ds = xr.open_dataset(path2, engine="netcdf4")
ds

<xarray.Dataset> Size: 444MB
Dimensions:             (nscans*10:MODIS_Swath_Type_GEO: 2030,
                         mframes:MODIS_Swath_Type_GEO: 1354,
                         nscans*20:MODIS_Swath_Type_GEO: 4060,
                         mframes*2:MODIS_Swath_Type_GEO: 2708, nscans: 203,
                         vecdim: 3, numqual: 4, numenc: 25, Scan Type: 10,
                         mframes: 1354, numbands: 37, fakeDim11: 2)
Coordinates:
    Scan Type           (nscans, Scan Type) |S1 2kB ...
Dimensions without coordinates: nscans*10:MODIS_Swath_Type_GEO,
                                mframes:MODIS_Swath_Type_GEO,
                                nscans*20:MODIS_Swath_Type_GEO,
                                mframes*2:MODIS_Swath_Type_GEO, nscans, vecdim,
                                numqual, numenc, mframes, numbands, fakeDim11
Data variables: (12/45)
    Latitude            (nscans*10:MODIS_Swath_Type_GEO, mframes:MODIS_Swath_Type_GEO) float32 11MB ...
    Longitude           (nscans*10:MODIS_Swath_Type_GEO, mframes:MODIS_Swath_Type_GEO) float32 11MB ...
    Scan Offset         (nscans*20:MODIS_Swath_Type_GEO, mframes*2:MODIS_Swath_Type_GEO) float64 88MB ...
    Track Offset        (nscans*20:MODIS_Swath_Type_GEO, mframes*2:MODIS_Swath_Type_GEO) float64 88MB ...
    Height Offset       (nscans*20:MODIS_Swath_Type_GEO, mframes*2:MODIS_Swath_Type_GEO) float64 88MB ...
    Height              (nscans*10:MODIS_Swath_Type_GEO, mframes:MODIS_Swath_Type_GEO) float32 11MB ...
    ...                  ...
    Focal_length        (numbands) float64 296B ...
    band_position       (numbands) float64 296B ...
    detector_space      (numbands) float64 296B ...
    detector_offsets    (numbands, fakeDim11) float64 592B ...
    T_offset            (numbands) float64 296B ...
    num_samples         (numbands) uint16 74B ...
Attributes: (12/29)
    HDFEOSVersion:                                           HDFEOS_V2.19
    StructMetadata.0:                                        GROUP=SwathStruc...
    band_number:                                             0
    Ephemeris Input Files.1:                                 GRANULE: http://...
    Ephemeris Input Files.2:                                 GRANULE: http://...
    Attitude Input Files.1:                                  GRANULE: http://...
    ...                                                      ...
    Cumulated gflags:                                        [0 0 0 0 0 0 0 0]
    utcpole File Header:                                     File Updated: 20...
    Polar Motion:                                            [ 0.080274  0.28...
    HDFEOS_FractionalOffset_nscans*20_MODIS_Swath_Type_GEO:  0.5
    HDFEOS_FractionalOffset_mframes*2_MODIS_Swath_Type_GEO:  0.0
    NOT EMPTY19622:                                          NOT EMPTY